# Confidence calibration

How well the model's stated `confidence` matches how often it is right, for one `evaluate` sweep.

After https://arxiv.org/pdf/2512.16030

In [43]:
%matplotlib ipympl

In [44]:
import pandas as pd

from analysis.build_data import load_trace_df

In [46]:
run_name = "sweep-20261105-1621"

In [47]:
# one row per forecast of every child run of the sweep (see analysis/build_data.py)
forecasts = load_trace_df("outcome-confidence", run_name)

In [48]:
forecasts

,run,model,system_prompt,exchange,category,event_type,market_title,response,as_of,predicted_probability,confidence,predicted_outcome,market_probability,outcome_occurred,resolution,state
0,claude-haiku-4-5-outcome-cutoff,claude-haiku-4-5,outcome-cutoff,polymarket,politics,unknown,Fauci extradited to Russia before April?,"Looking at this prediction market question, I ...",2025-01-24 08:00:00,None,0.97,False,0.0155,0.0,False,OK
1,claude-haiku-4-5-outcome-cutoff,claude-haiku-4-5,outcome-cutoff,polymarket,finance,None,Will Lam Research (LRCX) Q4 total revenue be a...,"Looking at this prediction market question, I ...",2026-07-19 00:00:05,None,0.48,False,0.5000,1.0,True,OK
2,claude-haiku-4-5-outcome-cutoff,claude-haiku-4-5,outcome-cutoff,polymarket,politics,social media post,Will Zelenskyy post 140-159 posts from May 5 t...,# Forecasting Zelenskyy's Post Count (May 5-12...,2026-05-03 00:00:06,None,0.38,False,0.1550,1.0,True,OK
3,claude-haiku-4-5-outcome-cutoff,claude-haiku-4-5,outcome-cutoff,kalshi,economics,None,Will the price of oil get below $90 by August ...,"Looking at this prediction market question, I ...",2022-08-09 04:00:00,None,0.87,True,0.0100,0.0,False,OK
4,claude-haiku-4-5-outcome-cutoff,claude-haiku-4-5,outcome-cutoff,polymarket,finance,None,Will Micron (MU) finish week of September 21 a...,"Looking at this prediction market question, I ...",2026-09-19 00:00:15,None,0.42,False,0.2300,1.0,True,OK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,claude-opus-5-5-outcome-cutoff,claude-opus-5-5,outcome-cutoff,polymarket,politics,speech quote,"Will Melania say ""AI Companies"" during AI talk...",Melania Trump's AI remarks have usually been s...,2026-01-16 08:00:00,None,0.92,False,0.2450,0.0,False,OK
196,claude-opus-5-5-outcome-cutoff,claude-opus-5-5,outcome-cutoff,polymarket,politics,social media post,Will CZ post 160-179 posts from May 8 to May 1...,"I can't look up the tracker data, so this esti...",2026-05-06 00:00:06,None,0.88,False,0.0950,1.0,True,OK
197,claude-opus-5-5-outcome-cutoff,claude-opus-5-5,outcome-cutoff,polymarket,politics,unknown,"Will Donald Trump's tie be red on May 2, 2026?","May 2, 2026 is a Saturday, and Trump often spe...",2026-05-01 00:00:04,None,0.84,False,0.3950,1.0,True,OK
198,claude-opus-5-5-outcome-cutoff,claude-opus-5-5,outcome-cutoff,polymarket,politics,speech quote,Will Trump claim victory by midnight?,Trump's victory speech in West Palm Beach bega...,2024-11-06 08:00:00,None,0.95,False,0.5000,0.0,False,OK


## Expected calibration error

ECE (Naeini et al., 2015) is the average gap between confidence and accuracy, with forecasts binned by their stated confidence into $B$ bins:

$$\mathrm{ECE} = \sum_{b=1}^{B} \frac{|B_b|}{N}\,\bigl|\mathrm{acc}(B_b) - \mathrm{conf}(B_b)\bigr|$$

Here a forecast is the outcome prompts' yes/no call, `confidence` is the model's stated probability that the call is right, and a forecast is accurate when the call matches `outcome_occurred`.

In [ ]:
# scoreable forecasts: a yes/no call with a confidence, on a market that has resolved
calibrated = forecasts.dropna(
    subset=["predicted_outcome", "confidence", "outcome_occurred"]
)

In [ ]:
calibrated["correct"] = calibrated["predicted_outcome"] == calibrated[
    "outcome_occurred"
].astype(bool)

In [ ]:
B = 10

In [ ]:
confidence_bins = pd.IntervalIndex.from_breaks([b / B for b in range(B + 1)])

In [ ]:
calibrated["confidence_bin"] = pd.cut(
    calibrated["confidence"], confidence_bins, include_lowest=True
)

In [ ]:
def expected_calibration_error(forecasts: pd.DataFrame) -> float:
    """sum_b |B_b|/N * |acc(B_b) - conf(B_b)| over the confidence bins."""
    bins = forecasts.groupby("confidence_bin", observed=True)
    acc = bins["correct"].mean()
    conf = bins["confidence"].mean()
    weight = bins.size() / len(forecasts)
    return (weight * (acc - conf).abs()).sum()

In [ ]:
ece = expected_calibration_error(calibrated)

In [ ]:
ece_by_model = calibrated.groupby("model").apply(
    expected_calibration_error, include_groups=False
)

In [ ]:
# the reliability table behind the number: per bin, its size, conf(B_b) and acc(B_b)
reliability = calibrated.groupby("confidence_bin", observed=True).agg(
    n=("correct", "size"), conf=("confidence", "mean"), acc=("correct", "mean")
)

In [ ]:
reliability

In [ ]:
ece_by_model